In [ ]:
# MCP SCENARIO: “Smart IT Helpdesk Assistant”
# Scenario Background

# You are working in a company called ABC Corp.

# Employees face issues like:

# VPN not working
# Printer not responding
# Software errors

# Instead of calling IT support, employees use an AI Helpdesk Bot.

#  What this Bot Should Do

# When a user types a problem:

# Understand the issue
# Decide if a ticket is needed
# Identify:
# Category (Network / Hardware / General)
# Priority (High / Medium)
# Create a ticket
# Show confirmation
# How MCP Fits Here
# Component	Role in Scenario
# Agent	Helpdesk Bot
# MCP Layer	Decision + Tool calling
# Tool	Ticket Creation System
# User	Employee



# STEP 0: DATABASE (Simulated storage)

tickets_db = []


# STEP 1: TOOL (MCP TOOL)

def create_ticket(issue, priority, category):
    """
    This function simulates a TOOL in MCP
    In real world → API / Database / ServiceNow
    """

    ticket_id = f"INC{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)

    return ticket


# STEP 2: AGENT REASONING (LLM SIMULATION)

def analyze_input(user_input):
    """
    Simulates how an LLM understands user input
    Extracts:
    - category
    - priority
    """

    text = user_input.lower()

    #  Category Detection
    if "vpn" in text:
        category = "network"
    elif "printer" in text:
        category = "hardware"
    elif "email" in text:
        category = "software"
    else:
        category = "general"

    #  Priority Detection
    if "urgent" in text or "immediately" in text:
        priority = "high"
    elif "slow" in text:
        priority = "low"
    else:
        priority = "medium"

    return category, priority



# STEP 3: DECISION ENGINE (MCP CORE)

def should_call_tool(user_input):
    """
    Decides whether to call a tool or not
    This is MCP decision layer
    """

    keywords = ["issue", "problem", "ticket", "not working"]

    return any(word in user_input.lower() for word in keywords)



# STEP 4: MCP ORCHESTRATOR

def mcp_agent(user_input):
    """
    This is the MAIN MCP FLOW
    It connects:
    Agent → Decision → Tool → Response
    """

    print("\nAgent received input:", user_input)

    # STEP 4.1: Decision
    if should_call_tool(user_input):

        print(" Decision: Tool call required")

        # STEP 4.2: Analyze input
        category, priority = analyze_input(user_input)

        print(f" Extracted → Category: {category}, Priority: {priority}")

        # STEP 4.3: Prepare payload (MCP format)
        payload = {
            "issue": user_input,
            "priority": priority,
            "category": category
        }

        print(" MCP Payload:", payload)

        # STEP 4.4: Call tool
        result = create_ticket(**payload)

        print(" Tool executed successfully")

        # STEP 4.5: Final response
        return f"""
         Ticket Created Successfully!

        Ticket ID: {result['ticket_id']}
        Issue: {result['issue']}
        Category: {result['category']}
        Priority: {result['priority']}
        """

    else:
        print(" Decision: No tool needed (AI response)")

        return " AI Response: Please describe your issue clearly."


# STEP 5: RUN INTERACTIVE LOOP

print(" MCP Demo Started (Type 'exit' to stop)\n")

while True:

    user_input = input("Enter your query: ")

    if user_input.lower() == "exit":
        print(" Exiting MCP demo...")
        break

    response = mcp_agent(user_input)
    print(response)


 MCP Demo Started (Type 'exit' to stop)

Enter your query: vpn not working

Agent received input: vpn not working
 Decision: Tool call required
 Extracted → Category: network, Priority: medium
 MCP Payload: {'issue': 'vpn not working', 'priority': 'medium', 'category': 'network'}
 Tool executed successfully

         Ticket Created Successfully!

        Ticket ID: INC1000
        Issue: vpn not working
        Category: network
        Priority: medium
        
Enter your query: printer not responding

Agent received input: printer not responding
 Decision: No tool needed (AI response)
 AI Response: Please describe your issue clearly.
Enter your query: software errors

Agent received input: software errors
 Decision: No tool needed (AI response)
 AI Response: Please describe your issue clearly.
Enter your query: network issue

Agent received input: network issue
 Decision: Tool call required
 Extracted → Category: general, Priority: medium
 MCP Payload: {'issue': 'network issue', 'pri

In [ ]:
# LLM Integration --same scenario as above

!pip install groq
import os
from groq import Groq
from google.colab import userdata

# Load API key securely
api_key = userdata.get("token")

client = Groq(api_key=api_key)

# STEP 0: DATABASE

tickets_db = []


# STEP 1: TOOL

def create_ticket(issue, priority, category):
    ticket_id = f"INC{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)
    return ticket



# STEP 2: LLM ANALYSIS (REPLACES RULES)

def analyze_with_llm(user_input):
    """
    LLM decides:
    - should_create_ticket
    - category
    - priority
    """

    prompt = f"""
You are an IT helpdesk assistant.

Analyze the user issue and respond in JSON format:

{{
  "create_ticket": true/false,
  "category": "network/hardware/software/general",
  "priority": "high/medium/low"
}}

User Input: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content

    try:
        import json
        parsed = json.loads(output)
    except:
        parsed = {
            "create_ticket": True,
            "category": "general",
            "priority": "medium"
        }

    return parsed



# STEP 3: MCP AGENT

def mcp_agent(user_input):

    print("\n Agent received:", user_input)

    # LLM Decision
    decision = analyze_with_llm(user_input)

    print(" LLM Decision:", decision)

    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "priority": decision["priority"],
            "category": decision["category"]
        }

        print(" MCP Payload:", payload)

        result = create_ticket(**payload)

        return f"""
 Ticket Created Successfully!

Ticket ID: {result['ticket_id']}
Issue: {result['issue']}
Category: {result['category']}
Priority: {result['priority']}
"""

    else:
        return " AI Response: No ticket required. Try basic troubleshooting."

# STEP 4: RUN LOOP

print(" LLM MCP Helpdesk Started (type 'exit')\n")

while True:

    user_input = input("Enter issue: ")

    if user_input.lower() == "exit":
        print(" Exiting...")
        break

    response = mcp_agent(user_input)
    print(response)



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 10.0 MB/s eta 0:00:00
 LLM MCP Helpdesk Started (type 'exit')

Enter issue: vpn issue

 Agent received: vpn issue
 LLM Decision: {'create_ticket': True, 'category': 'network', 'priority': 'medium'}
 MCP Payload: {'issue': 'vpn issue', 'priority': 'medium', 'category': 'network'}

 Ticket Created Successfully!

Ticket ID: INC1000
Issue: vpn issue
Category: network
Priority: medium

Enter issue: software issue

 Agent received: software issue
 LLM Decision: {'create_ticket': True, 'category': 'software', 'priority': 'medium'}
 MCP Payload: {'issue': 'software issue', 'priority': 'medium', 'category': 'software'}

 Ticket Created Successfully!

Ticket ID: INC1001
Issue: software issue
Category: software
Priority: medium

Enter issue: network issue

 Agent received: network issue
 LLM Decision: {'create_ticket': True, 'category': 'network', 'priority': 'medium'}
 MCP Payload: {'issue': 'network issue', 'priority': 'medium', 'catego

In [ ]:
# MCP SCENARIO: “Smart HR Onboarding Assistant”
#  Scenario Background
# You are working in a company called XYZ Corp.
# New employees often face challenges during onboarding, such as:
# - Trouble accessing payroll portal
# - Confusion about leave policies
# - Difficulty setting up email accounts
# - Questions about training schedules
#  Instead of emailing HR or waiting for responses, employees use an AI Onboarding Bot.

#  What this Bot Should Do
# When a new hire types a question/problem:
# - Understand the query (e.g., “I can’t log into payroll”)
# - Decide if escalation to HR is needed
# - Identify:
# - Category (Payroll / Policy / IT Setup / Training)
# - Priority (High / Medium)
# - Create a support ticket if required
# - Provide instant guidance (FAQs, step-by-step instructions)
# - Show confirmation and next steps

#  How MCP Fits Here
# |  |  |
# |  |  |
# |  |  |
# |  |  |
# |  |  |



# This way, the MCP framework is reused in a Human Resources context, where the AI assistant streamlines onboarding, reduces HR workload, and ensures employees feel supported from day one.
# Would you like me to design another variation in a customer service setting (like retail or banking), so you can compare how MCP adapts across industries?

# ==========================================
# STEP 0: DATABASE
# ==========================================
tickets_db = []


# ==========================================
# STEP 1: TOOL
# ==========================================
def create_ticket(issue, priority, category):

    ticket_id = f"HR{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category,
        "status": "Open"
    }

    tickets_db.append(ticket)

    return ticket


# ==========================================
# STEP 2: ANALYZE INPUT
# ==========================================
def analyze_input(user_input):

    text = user_input.lower()

    # CATEGORY
    if "payroll" in text or "salary" in text:
        category = "Payroll"
    elif "leave" in text or "policy" in text:
        category = "Policy"
    elif "email" in text or "login" in text or "account" in text:
        category = "IT Setup"
    elif "training" in text or "schedule" in text:
        category = "Training"
    else:
        category = "General"

    # PRIORITY
    if "urgent" in text or "immediately" in text:
        priority = "High"
    elif "not working" in text or "can't" in text or "cannot" in text or "unable" in text:
        priority = "High"
    elif "help" in text or "confused" in text:
        priority = "Medium"
    else:
        priority = "Low"

    return category, priority


# ==========================================
# STEP 3: DECISION ENGINE (FIXED)
# ==========================================
def should_call_tool(user_input):

    text = user_input.lower()

    keywords = [
        "not working", "issue", "problem",
        "can't", "cannot", "unable",
        "error", "fail", "login issue"
    ]

    return any(word in text for word in keywords)


# ==========================================
# STEP 4: RESPONSE GENERATOR
# ==========================================
def generate_response(category):

    responses = {
        "Payroll": "Check payroll portal or reset password.",
        "Policy": "Visit HR portal → Policies section.",
        "IT Setup": "Try password reset or contact IT.",
        "Training": "Check training dashboard.",
        "General": "HR will assist you."
    }

    return responses.get(category)


# ==========================================
# STEP 5: MCP ORCHESTRATOR
# ==========================================
def mcp_agent(user_input):

    print("\nAgent received input:", user_input)

    category, priority = analyze_input(user_input)

    print(f" Extracted → Category: {category}, Priority: {priority}")

    # DECISION
    if should_call_tool(user_input):

        print(" Decision: Tool call required")

        payload = {
            "issue": user_input,
            "priority": priority,
            "category": category
        }

        print(" MCP Payload:", payload)

        result = create_ticket(**payload)

        print(" Tool executed successfully")

        return f"""
 Ticket Created Successfully!

 Ticket ID: {result['ticket_id']}
 Category: {result['category']}
 Priority: {result['priority']}
 Status: {result['status']}
 """

    else:
        print(" Decision: No tool needed")

        response = generate_response(category)

        return f"""
 Instant Help:

 Category: {category}
 Priority: {priority}
 Response: {response}
 """


# ==========================================
# STEP 6: RUN LOOP
# ==========================================
print(" HR MCP Demo Started (Type 'exit')\n")

while True:

    user_input = input("Enter query: ")

    if user_input.lower() == "exit":
        break

    print(mcp_agent(user_input))

 HR MCP Demo Started (Type 'exit')

Enter query: I cant login to payroll

Agent received input: I cant login to payroll
 Extracted → Category: Payroll, Priority: Low
 Decision: No tool needed

 Instant Help:

 Category: Payroll
 Priority: Low
 Response: Check payroll portal or reset password.
 
Enter query: can you immediately help me with login issue? 

Agent received input: can you immediately help me with login issue? 
 Extracted → Category: IT Setup, Priority: High
 Decision: Tool call required
 MCP Payload: {'issue': 'can you immediately help me with login issue? ', 'priority': 'High', 'category': 'IT Setup'}
 Tool executed successfully

 Ticket Created Successfully!

 Ticket ID: HR1000
 Category: IT Setup
 Priority: High
 Status: Open
 
Enter query: exit


In [ ]:
# LLM Integration

import os
from groq import Groq
from google.colab import userdata

# Load API key securely
api_key = userdata.get("token")

client = Groq(api_key=api_key)


# ==========================================
# STEP 0: DATABASE
# ==========================================

tickets_db = []


# ==========================================
# STEP 1: TOOL
# ==========================================

def create_ticket(issue, priority, category):

    ticket_id = f"HR{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category,
        "status": "Open"
    }

    tickets_db.append(ticket)
    return ticket


# ==========================================
# STEP 2: LLM ANALYSIS (HR CONTEXT)
# ==========================================

def analyze_with_llm(user_input):
    """
    LLM decides:
    - create_ticket
    - category
    - priority
    """

    prompt = f"""
You are an HR onboarding assistant for a company.

Analyze the employee query and respond ONLY in JSON:

{{
  "create_ticket": true/false,
  "category": "Payroll/Policy/IT Setup/Training/General",
  "priority": "High/Medium/Low"
}}

Rules:
- Payroll issues → High
- Login/email issues → High
- Policy questions → Medium or Low
- Training queries → Low
- Create ticket ONLY if problem/issue exists

User Input: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content

    try:
        import json
        parsed = json.loads(output)
    except:
        parsed = {
            "create_ticket": True,
            "category": "General",
            "priority": "Medium"
        }

    return parsed


# ==========================================
# STEP 3: FAQ RESPONSE
# ==========================================

def generate_response(category):

    responses = {
        "Payroll": "Please check payroll portal or reset credentials.",
        "Policy": "Visit HR portal → Policies section.",
        "IT Setup": "Try password reset or contact IT support.",
        "Training": "Training schedule is available on dashboard.",
        "General": "HR team will assist you."
    }

    return responses.get(category)


# ==========================================
# STEP 4: MCP AGENT
# ==========================================

def mcp_agent(user_input):

    print("\n Agent received:", user_input)

    # LLM Decision
    decision = analyze_with_llm(user_input)

    print(" LLM Decision:", decision)

    category = decision["category"]
    priority = decision["priority"]

    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "priority": priority,
            "category": category
        }

        print(" MCP Payload:", payload)

        result = create_ticket(**payload)

        return f"""
 Ticket Created Successfully!

 Ticket ID: {result['ticket_id']}
 Category: {result['category']}
 Priority: {result['priority']}
 Status: {result['status']}
 """

    else:
        response = generate_response(category)

        return f"""
 Instant Help:

 Category: {category}
 Priority: {priority}
 Response: {response}
 """


# ==========================================
# STEP 5: RUN LOOP
# ==========================================

print(" HR LLM MCP Assistant Started (type 'exit')\n")

while True:

    user_input = input("Enter query: ")

    if user_input.lower() == "exit":
        print(" Exiting...")
        break

    response = mcp_agent(user_input)
    print(response)


 HR LLM MCP Assistant Started (type 'exit')

Enter query: can you immediately help me with login issue ?

 Agent received: can you immediately help me with login issue ?
 LLM Decision: {'create_ticket': True, 'category': 'IT Setup', 'priority': 'High'}
 MCP Payload: {'issue': 'can you immediately help me with login issue ?', 'priority': 'High', 'category': 'IT Setup'}

 Ticket Created Successfully!

 Ticket ID: HR1000
 Category: IT Setup
 Priority: High
 Status: Open
 
Enter query: facing dificulty in training schedules

 Agent received: facing dificulty in training schedules
 LLM Decision: {'create_ticket': True, 'category': 'General', 'priority': 'Medium'}
 MCP Payload: {'issue': 'facing dificulty in training schedules', 'priority': 'Medium', 'category': 'General'}

 Ticket Created Successfully!

 Ticket ID: HR1001
 Category: General
 Priority: Medium
 Status: Open
 
Enter query: exit
 Exiting...


In [ ]:
# MCP SCENARIO: “Smart Banking Support Assistant”
#  Scenario Background
# You are working in a company called FinTrust Bank.
# Customers often face issues such as:
# - Credit card not working
# - Trouble with online banking login
# - Queries about loan status
# - Transaction disputes
#  Instead of calling customer care, customers use an AI Banking Support Bot.

#  What this Bot Should Do
# When a customer types a problem:
# - Understand the issue (e.g., “My card was declined”)
# - Decide if escalation to a human agent is needed
# - Identify:
# - Category (Card Services / Online Banking / Loans / Transactions)
# - Priority (High / Medium)
# - Create a support ticket if required
# - Provide instant guidance (FAQs, troubleshooting steps, policy info)
# - Show confirmation and next steps

#  How MCP Fits Here
# |  |  |
# |  |  |
# |  |  |
# |  |  |
# |  |  |



# This way, MCP is applied in a financial services context, where the AI assistant reduces call center load, provides quick resolutions, and ensures customers feel supported with secure, reliable guidance.
# Would you like me to craft one more in a healthcare setting (like hospital patient support), so you can see how MCP adapts to critical service environments?

# ==========================================
# STEP 0: DATABASE (Simulated storage)
# ==========================================

tickets_db = []


# ==========================================
# STEP 1: TOOL (MCP TOOL)
# ==========================================

def create_ticket(issue, priority, category):

    ticket_id = f"BNK{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category,
        "status": "Open"
    }

    tickets_db.append(ticket)
    return ticket


# ==========================================
# STEP 2: AGENT REASONING (NO LLM)
# ==========================================

def analyze_input(user_input):
    """
    Extract:
    - category
    - priority
    """

    text = user_input.lower()

    # CATEGORY DETECTION
    if "card" in text or "credit" in text or "debit" in text:
        category = "Card Services"
    elif "login" in text or "password" in text or "account" in text:
        category = "Online Banking"
    elif "loan" in text or "emi" in text:
        category = "Loans"
    elif "transaction" in text or "payment" in text or "fraud" in text:
        category = "Transactions"
    else:
        category = "General"

    # PRIORITY DETECTION
    if "urgent" in text or "fraud" in text or "not working" in text:
        priority = "High"
    elif "issue" in text or "problem" in text:
        priority = "Medium"
    else:
        priority = "Low"

    return category, priority


# ==========================================
# STEP 3: DECISION ENGINE (MCP CORE)
# ==========================================

def should_call_tool(user_input):

    keywords = [
        "not working", "issue", "problem",
        "failed", "declined", "fraud",
        "error", "unable"
    ]

    return any(word in user_input.lower() for word in keywords)


# ==========================================
# STEP 4: RESPONSE GENERATOR (FAQ)
# ==========================================

def generate_response(category):

    responses = {
        "Card Services": "Check if your card is active and has sufficient balance.",
        "Online Banking": "Try resetting your password using 'Forgot Password'.",
        "Loans": "Loan status is available in your banking dashboard.",
        "Transactions": "Check transaction history or contact support for disputes.",
        "General": "Customer support will assist you shortly."
    }

    return responses.get(category)


# ==========================================
# STEP 5: MCP ORCHESTRATOR
# ==========================================

def mcp_agent(user_input):

    print("\n Agent received:", user_input)

    # Analyze input
    category, priority = analyze_input(user_input)

    print(f" Extracted → Category: {category}, Priority: {priority}")

    # Decision
    if should_call_tool(user_input):

        print(" Decision: Tool call required")

        payload = {
            "issue": user_input,
            "priority": priority,
            "category": category
        }

        print(" MCP Payload:", payload)

        result = create_ticket(**payload)

        print(" Tool executed successfully")

        return f"""
 Ticket Created Successfully!

 Ticket ID: {result['ticket_id']}
 Category: {result['category']}
 Priority: {result['priority']}
 Status: {result['status']}

 Bank support will contact you soon.
 """

    else:
        print(" Decision: No tool needed")

        response = generate_response(category)

        return f"""
 Instant Help:

 Category: {category}
 Priority: {priority}
 Response: {response}
 """


# ==========================================
# STEP 6: RUN LOOP
# ==========================================

print(" Banking MCP Assistant Started (type 'exit')\n")

while True:

    user_input = input("Enter your query: ")

    if user_input.lower() == "exit":
        print(" Exiting...")
        break

    response = mcp_agent(user_input)
    print(response)



 Banking MCP Assistant Started (type 'exit')

Enter your query: credit card not working

 Agent received: credit card not working
 Extracted → Category: Card Services, Priority: High
 Decision: Tool call required
 MCP Payload: {'issue': 'credit card not working', 'priority': 'High', 'category': 'Card Services'}
 Tool executed successfully

 Ticket Created Successfully!

 Ticket ID: BNK1000
 Category: Card Services
 Priority: High
 Status: Open

 Bank support will contact you soon.
 
Enter your query: queries about loan status

 Agent received: queries about loan status
 Extracted → Category: Loans, Priority: Low
 Decision: No tool needed

 Instant Help:

 Category: Loans
 Priority: Low
 Response: Loan status is available in your banking dashboard.
 
Enter your query: exit
 Exiting...


In [ ]:
# LLM Integration

import os
from groq import Groq
from google.colab import userdata

# Load API key securely
api_key = userdata.get("token")

client = Groq(api_key=api_key)


# ==========================================
# STEP 0: DATABASE
# ==========================================

tickets_db = []


# ==========================================
# STEP 1: TOOL
# ==========================================

def create_ticket(issue, priority, category):

    ticket_id = f"BNK{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category,
        "status": "Open"
    }

    tickets_db.append(ticket)
    return ticket


# ==========================================
# STEP 2: LLM ANALYSIS (BANKING CONTEXT)
# ==========================================

def analyze_with_llm(user_input):
    """
    LLM decides:
    - create_ticket
    - category
    - priority
    """

    prompt = f"""
You are a banking support assistant for FinTrust Bank.

Analyze the customer query and respond ONLY in JSON:

{{
  "create_ticket": true/false,
  "category": "Card Services/Online Banking/Loans/Transactions/General",
  "priority": "High/Medium/Low"
}}

Rules:
- Card declined / fraud → High
- Login issues → High
- Loan queries → Medium/Low
- General info → No ticket
- Create ticket ONLY if problem/issue exists

User Input: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content

    try:
        import json
        parsed = json.loads(output)
    except:
        parsed = {
            "create_ticket": True,
            "category": "General",
            "priority": "Medium"
        }

    return parsed


# ==========================================
# STEP 3: FAQ RESPONSE
# ==========================================

def generate_response(category):

    responses = {
        "Card Services": "Ensure your card is active and has sufficient balance.",
        "Online Banking": "Try resetting password using 'Forgot Password'.",
        "Loans": "Check your loan status in the dashboard.",
        "Transactions": "Review transaction history or raise dispute.",
        "General": "Bank support will assist you."
    }

    return responses.get(category)


# ==========================================
# STEP 4: MCP AGENT
# ==========================================

def mcp_agent(user_input):

    print("\n Agent received:", user_input)

    # LLM Decision
    decision = analyze_with_llm(user_input)

    print(" LLM Decision:", decision)

    category = decision["category"]
    priority = decision["priority"]

    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "priority": priority,
            "category": category
        }

        print(" MCP Payload:", payload)

        result = create_ticket(**payload)

        return f"""
 Ticket Created Successfully!

 Ticket ID: {result['ticket_id']}
 Category: {result['category']}
 Priority: {result['priority']}
 Status: {result['status']}

 Bank support team will contact you soon.
 """

    else:
        response = generate_response(category)

        return f"""
 Instant Help:

 Category: {category}
 Priority: {priority}
 Response: {response}
 """


# ==========================================
# STEP 5: RUN LOOP
# ==========================================

print(" Banking LLM MCP Assistant Started (type 'exit')\n")

while True:

    user_input = input("Enter your query: ")

    if user_input.lower() == "exit":
        print(" Exiting...")
        break

    response = mcp_agent(user_input)
    print(response)



 Banking LLM MCP Assistant Started (type 'exit')

Enter your query: credit card not working

 Agent received: credit card not working
 LLM Decision: {'create_ticket': True, 'category': 'Card Services', 'priority': 'High'}
 MCP Payload: {'issue': 'credit card not working', 'priority': 'High', 'category': 'Card Services'}

 Ticket Created Successfully!

 Ticket ID: BNK1000
 Category: Card Services
 Priority: High
 Status: Open

 Bank support team will contact you soon.
 
Enter your query: can you help me with password issue ?

 Agent received: can you help me with password issue ?
 LLM Decision: {'create_ticket': True, 'category': 'Online Banking', 'priority': 'High'}
 MCP Payload: {'issue': 'can you help me with password issue ?', 'priority': 'High', 'category': 'Online Banking'}

 Ticket Created Successfully!

 Ticket ID: BNK1001
 Category: Online Banking
 Priority: High
 Status: Open

 Bank support team will contact you soon.
 
Enter your query: loan queries

 Agent received: loan que

In [2]:
#Create a  Weather Tool MCP Server that any AI agent can use with sample use case
# ============================================
# INSTALL
# ============================================

!pip install groq gradio

# ============================================
# IMPORTS
# ============================================

import json
import re
import gradio as gr
from groq import Groq
from google.colab import userdata

# Load API key
api_key = userdata.get("token")
client = Groq(api_key=api_key)

# ============================================
# TOOL 1: WEATHER
# ============================================

weather_data = {
    "delhi": "32°C, Sunny",
    "mumbai": "28°C, Humid",
    "london": "15°C, Cloudy"
}

def get_weather(city):
    return weather_data.get(city.lower(), "Weather data not available")

# ============================================
# TOOL 2: NEWS
# ============================================

news_data = {
    "india": "India: Tech sector growing rapidly ",
    "world": "Global markets show mixed trends ",
    "sports": "India wins thrilling cricket match "
}

def get_news(topic):
    return news_data.get(topic.lower(), "No news available")

# ============================================
# JSON EXTRACTOR
# ============================================

def extract_json(text):
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except:
            return None
    return None

# ============================================
# LLM DECISION (MCP BRAIN)
# ============================================

def analyze_with_llm(user_input):

    prompt = f"""
You are an AI assistant with tools.

Decide which tool to use.

Return ONLY JSON:

{{
  "tool": "weather/news/none",
  "input": "city or topic",
  "response": "normal reply if no tool"
}}

User Query: "{user_input}"
"""

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        output = response.choices[0].message.content
        parsed = extract_json(output)

    except Exception as e:
        print("Error:", e)
        parsed = None

    if not parsed:
        parsed = {
            "tool": "none",
            "input": "",
            "response": "Sorry, I couldn't understand."
        }

    return parsed

# ============================================
# MCP AGENT
# ============================================

def mcp_multi_tool_agent(user_input):

    decision = analyze_with_llm(user_input)

    tool = decision.get("tool")
    tool_input = decision.get("input")

    # 🔹 WEATHER TOOL
    if tool == "weather":
        if not tool_input:
            return " Please specify a city."
        result = get_weather(tool_input)
        return f" Weather in {tool_input.title()}: {result}"

    # 🔹 NEWS TOOL
    elif tool == "news":
        if not tool_input:
            return " Please specify a topic (india/world/sports)."
        result = get_news(tool_input)
        return f" News ({tool_input.title()}): {result}"

    # 🔹 NO TOOL
    else:
        return decision.get("response")


# ============================================
# GRADIO UI
# ============================================

def chatbot_response(message, history):
    return mcp_multi_tool_agent(message)

chat_ui = gr.ChatInterface(
    fn=chatbot_response,
    title=" Multi-Tool MCP Assistant",
    description="Ask weather or news (e.g., 'Weather in Delhi', 'News about India')"
)

chat_ui.launch()

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0bb30c91ef3e3aa8c8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
